# 03. 학습

모델 선택 → 하이퍼파라미터 설정 → 학습 → WandB / Notion 기록

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')

print('환경 설정 완료')

---
## 1. 모델 선택

| 모델 | VRAM | 특징 |
|---|---|---|
| `unsloth/Qwen3-14B` | ~20GB | 베이스라인, 한국어 강함 |
| `unsloth/Qwen3-8B` | ~12GB | 경량, 빠른 실험 |
| `unsloth/gemma-3-12b-it` | ~16GB | 다국어 |
| `unsloth/Phi-4` | ~10GB | 소형 고성능 |
| `unsloth/llama-3.1-8b-instruct` | ~12GB | Meta Llama |

In [ ]:
from src.training.trainer import ModelConfig, LoraConfig, TrainConfig, ExperimentConfig, HyperParams

# ============================================================
#  모델 선택
# ============================================================
model_cfg = ModelConfig(
    model_name     = "unsloth/Qwen3-14B",   # ← 여기서 모델 변경
    max_seq_length = 2048,
    load_in_4bit   = True,                  # QLoRA 4-bit 양자화
)

print(f'선택 모델: {model_cfg.model_name}')

---
## 2. 하이퍼파라미터 설정

### LoRA 파라미터 가이드
| r | 학습 파라미터 | 메모리 | 권장 상황 |
|---|---|---|---|
| 8 | 적음 | 낮음 | 빠른 실험 |
| 16 | 보통 | 보통 | 기본값 |
| 32 | 많음 | 높음 | 성능 개선 필요 시 |
| 64 | 최대 | 매우 높음 | 최고 성능 목표 |

### 학습률 가이드
| lr | 특징 |
|---|---|
| 1e-4 ~ 3e-4 | LoRA 표준 범위 |
| 2e-4 | 베이스라인 기본값 |
| 5e-5 | 보수적, 안정적 |


In [ ]:
# ============================================================
#  하이퍼파라미터 설정  ← 여기를 수정하세요
# ============================================================
lora_cfg = LoraConfig(
    r       = 16,       # LoRA rank: 8 / 16 / 32 / 64
    alpha   = 16,       # 보통 r과 동일하게 설정
    dropout = 0.0,
)

train_cfg = TrainConfig(
    num_epochs     = 3,
    batch_size     = 1,      # per_device_train_batch_size
    grad_accum     = 32,     # 실효 배치 = batch_size × grad_accum
    learning_rate  = 2e-4,
    lr_scheduler   = "cosine",   # cosine / linear / constant
    warmup_ratio   = 0.05,
    bf16           = True,
    response_only  = True,       # True: 요약 부분만 loss 계산 (권장)
    max_new_tokens = 256,
    seed           = 42,
)

# ============================================================
#  실험 정보  ← 매 실험마다 수정하세요
# ============================================================
exp_cfg = ExperimentConfig(
    run_name  = "qwen3-14b-lora-r16-lr2e4",       # 실험명 (Notion 제목)
    dataset   = "DialogSum-KO",
    purpose   = "베이스라인 SFT 재현 실험",           # 실험 목적
    tags      = ["LoRA", "Qwen3", "baseline"],
    output_dir = "../outputs/checkpoints",
)

hp = HyperParams(model=model_cfg, lora=lora_cfg, train=train_cfg, experiment=exp_cfg)
hp.summary()

---
## 3. 프롬프트 템플릿

In [ ]:
PROMPT_TEMPLATE = """당신은 대화 내용을 간결하게 요약하는 AI입니다.

[대화]
{dialogue}

[요약]
{summary}"""

# 프롬프트 샘플 확인
sample_dialogue = "#Person1#: 안녕하세요.\n#Person2#: 반갑습니다."
print(PROMPT_TEMPLATE.format(dialogue=sample_dialogue, summary=""))

---
## 4. WandB / Notion 연동

In [ ]:
import yaml
from src.utils.wandb_logger import WandbLogger
from src.utils.notion_logger import NotionLogger

with open('../configs/base_config.yaml') as f:
    base_config = yaml.safe_load(f)

# WandB 설정 - run_name 실험명으로 오버라이드
base_config['wandb']['name'] = exp_cfg.run_name
base_config['wandb']['tags'] = exp_cfg.tags

wandb_logger  = WandbLogger(base_config)
notion_logger = NotionLogger()

print('WandB / Notion 연동 완료')
print(f'WandB run URL: {wandb_logger.run.url}')

---
## 5. 데이터 로드

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
dev_df   = pd.read_csv('../data/processed/dev.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

print(f'train: {len(train_df):,}  |  dev: {len(dev_df):,}  |  test: {len(test_df):,}')

---
## 6. 학습 실행

In [ ]:
from src.training.trainer import DialogueSummarizationTrainer

trainer = DialogueSummarizationTrainer(
    hp              = hp,
    prompt_template = PROMPT_TEMPLATE,
    notion_logger   = notion_logger,
    wandb_logger    = wandb_logger,
)

rouge_scores = trainer.run(train_df, dev_df)

print('\n=== 최종 ROUGE 점수 ===')
for k, v in rouge_scores.items():
    print(f'  {k}: {v}')

---
## 7. 제출 파일 생성

In [ ]:
from unsloth import FastLanguageModel
import torch
from src.inference.submit import SubmissionGenerator

FastLanguageModel.for_inference(trainer.model)

predictions = []
for dialogue in test_df['dialogue']:
    prompt = PROMPT_TEMPLATE.format(dialogue=dialogue, summary="")
    inputs = trainer.tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=hp.train.max_new_tokens,
            do_sample=False,
        )
    pred = trainer.tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    predictions.append(pred)

gen = SubmissionGenerator(
    sample_submission_path='../data/raw/sample_submission.csv',
    output_dir='../outputs/predictions',
)
submit_path = gen.save(predictions=predictions, filename=f"{exp_cfg.run_name}.csv")
print(f'제출 파일: {submit_path}')

---
## 8. 마무리

In [ ]:
wandb_logger.finish()
print('완료!')
print(f'WandB : {wandb_logger.run.url}')